# 🏠 House Price Regression — PyTorch + ONNX Deployment
**Dataset:** California Housing (sklearn built-in) 
**Task:** Regression — predict median house value 
**Deployment:** ONNX → GitHub Pages

## Step 1: Install Libraries

In [ ]:
!pip install onnx onnxruntime skl2onnx

## Step 2: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import Adam
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import onnx
import onnxruntime as ort
print('All libraries loaded!')

## Step 3: Hyperparameters

In [ ]:
BATCH_SIZE   = 32
LEARNING_RATE = 0.001
EPOCHS       = 150
RANDOM_SEED  = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Step 4: Load & Explore the Dataset
The **California Housing** dataset from sklearn contains 8 features describing California census block groups, and the target is the **median house value** (in $100,000s).

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print('Dataset shape:', df.shape)
print('\nFeature names:', housing.feature_names)
print('\nTarget: MedHouseVal (median house value in $100k)')
df.head()

In [ ]:
df.describe()

## Step 5: Preprocess Data

In [ ]:
# Separate features and target
X = df[housing.feature_names].values   # shape: (20640, 8)
y = df['MedHouseVal'].values            # shape: (20640,)

print('X shape:', X.shape)
print('y shape:', y.shape)
print('y range: [{:.2f}, {:.2f}]'.format(y.min(), y.max()))

In [ ]:
# Train/test split — 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)
print('Train:', X_train.shape, '| Test:', X_test.shape)

In [ ]:
# Convert to float32
X_train = X_train.astype(np.float32)
X_test  = X_test.astype(np.float32)
y_train = y_train.astype(np.float32)
y_test  = y_test.astype(np.float32)

# Compute mean & std from TRAINING set only (to avoid data leakage)
X_means = X_train.mean(axis=0)
X_stds  = X_train.std(axis=0) + 1e-8   # epsilon to avoid division by zero

print('Feature means:', np.round(X_means, 3))
print('Feature stds: ', np.round(X_stds,  3))

In [ ]:
# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train)
X_test_t  = torch.tensor(X_test)
y_train_t = torch.tensor(y_train).unsqueeze(1)  # shape: (N, 1)
y_test_t  = torch.tensor(y_test).unsqueeze(1)

print('X_train_t:', X_train_t.shape)
print('y_train_t:', y_train_t.shape)

In [ ]:
# Build DataLoader using TensorDataset
train_ds     = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
print('DataLoader ready — batches per epoch:', len(train_loader))

## Step 6: Define the Deep Neural Network Architecture

Architecture: **8 → 64 → 32 → 16 → 1**  
- Input: 8 features  
- Three hidden layers with ReLU activations  
- Dropout for regularization  
- Output: 1 continuous value (predicted house price)

In [ ]:
class HousePriceDLNet(nn.Module):
    def __init__(self, x_means, x_stds):
        super(HousePriceDLNet, self).__init__()

        # Store normalization stats as non-trainable buffers
        self.register_buffer('x_means', torch.tensor(x_means, dtype=torch.float32))
        self.register_buffer('x_stds',  torch.tensor(x_stds,  dtype=torch.float32))

        # Architecture: 8 -> 64 -> 32 -> 16 -> 1
        self.linear1 = nn.Linear(8,  64)
        self.act1    = nn.ReLU()
        self.drop1   = nn.Dropout(0.2)

        self.linear2 = nn.Linear(64, 32)
        self.act2    = nn.ReLU()
        self.drop2   = nn.Dropout(0.2)

        self.linear3 = nn.Linear(32, 16)
        self.act3    = nn.ReLU()

        self.linear4 = nn.Linear(16, 1)   # output layer — no activation (regression)

    def forward(self, x):
        # Standardize input
        x = (x - self.x_means) / self.x_stds

        # Forward pass
        x = self.drop1(self.act1(self.linear1(x)))
        x = self.drop2(self.act2(self.linear2(x)))
        x = self.act3(self.linear3(x))
        x = self.linear4(x)
        return x


model = HousePriceDLNet(X_means, X_stds)
print(model)

## Step 7: Train the Model

In [ ]:
optimizer = Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn   = nn.MSELoss()   # MSE for regression (not cross-entropy)

loss_history = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss   = loss_fn(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    loss_history.append(avg_loss)

    if (epoch + 1) % 25 == 0:
        print(f'Epoch [{epoch+1:3d}/{EPOCHS}]  MSE Loss: {avg_loss:.4f}')

print('\nTraining complete!')

## Step 8: Evaluate the Model

In [ ]:
model.eval()
with torch.no_grad():
    y_pred_test = model(X_test_t)

y_pred_np = y_pred_test.detach().numpy()
y_test_np = y_test_t.numpy()

r2 = r2_score(y_test_np, y_pred_np)
print(f'R² Score on Test Set: {r2:.4f}')

# Show a few sample predictions
print('\nSample Predictions (in $100k):')
print(f'{"Predicted":>12}  {"Actual":>8}')
for i in range(10):
    print(f'{y_pred_np[i][0]:>12.2f}  {y_test_np[i][0]:>8.2f}')

## Step 9: Export Model to ONNX
The ONNX file embeds the model weights and architecture. We load it in JavaScript for browser-side inference.

In [ ]:
model.eval()

# Dummy input: 1 sample, 8 features
dummy_input = torch.zeros(1, 8, dtype=torch.float32)

onnx_filename = 'house_price_model.onnx'

torch.onnx.export(
    model,
    dummy_input,
    onnx_filename,
    export_params   = True,
    opset_version   = 15,          # must match the JS ONNX runtime version
    do_constant_folding = True,
    input_names     = ['input'],
    output_names    = ['output'],
    dynamic_axes    = {'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

print(f'Model exported to: {onnx_filename}')

# Verify the exported model
onnx_model = onnx.load(onnx_filename)
onnx.checker.check_model(onnx_model)
print('ONNX model verified successfully!')

In [ ]:
# Quick sanity check: run the ONNX model locally
sess = ort.InferenceSession(onnx_filename)
sample = X_test[:1].astype(np.float32)
onnx_out = sess.run(['output'], {'input': sample})
torch_out = model(torch.tensor(sample)).item()

print(f'ONNX output:  {onnx_out[0][0][0]:.4f}')
print(f'PyTorch output: {torch_out:.4f}')
print(f'Actual value:   {y_test[0]:.4f}')
print('\nONNX output matches PyTorch — ready for deployment!')

## Step 10: Download the ONNX File
Run this cell if using **Google Colab** to download the file to your computer.

In [ ]:
try:
    from google.colab import files
    files.download('house_price_model.onnx')
    print('Download started!')
except ImportError:
    print('Not running in Colab — find house_price_model.onnx in your working directory.')

---
## Feature Reference (for the Website)

| # | Feature | Description | Typical Range |
|---|---------|-------------|---------------|
| 1 | MedInc | Median income (in $10k) | 0.5 – 15 |
| 2 | HouseAge | Median house age (years) | 1 – 52 |
| 3 | AveRooms | Average rooms per household | 1 – 10 |
| 4 | AveBedrms | Average bedrooms per household | 1 – 5 |
| 5 | Population | Block group population | 3 – 35,682 |
| 6 | AveOccup | Average household occupancy | 1 – 10 |
| 7 | Latitude | Block group latitude | 32 – 42 |
| 8 | Longitude | Block group longitude | -124 – -114 |

**Output:** Predicted median house value in **$100,000s**  
*(e.g., output of 2.5 means ~$250,000)*